# **CHEST XRAY**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

100%|██████████| 2.29G/2.29G [00:23<00:00, 103MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2


In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from skimage import filters
import time

# Base path for dataset
BASE_PATH = '/root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2'

# Function to verify the existence of dataset paths
def verify_paths():
    categories = ['NORMAL', 'PNEUMONIA']
    for split in ['train', 'test', 'val']:
        for category in categories:
            path = os.path.join(BASE_PATH, f'chest_xray/{split}', category)
            if not os.path.exists(path):
                print(f"Warning: Path does not exist: {path}")
            else:
                print(f"Found directory: {path}")
                print(f"Number of images: {len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])}")

# Function to load images from a given directory
def load_xray_data(img_size=(128, 128)):
    categories = ['NORMAL', 'PNEUMONIA']

    def load_images_from_path(split_path, img_size):
        images = []
        labels = []
        for category in categories:
            category_path = os.path.join(split_path, category)
            if not os.path.exists(category_path):
                print(f"Warning: Path does not exist: {category_path}")
                continue

            print(f"Loading {category} images from {category_path}")
            image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in image_files:
                img_path = os.path.join(category_path, img_name)
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Failed to load image: {img_path}")
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    img = cv2.resize(img, img_size)
                    images.append(img)
                    labels.append(category)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

            print(f"Loaded {len(images)} images for {category}")

        return np.array(images), np.array(labels)

    print("\nLoading training data...")
    X_train, y_train = load_images_from_path(os.path.join(BASE_PATH, 'chest_xray/train'), img_size)

    print("\nLoading validation data...")
    X_val, y_val = load_images_from_path(os.path.join(BASE_PATH, 'chest_xray/val'), img_size)

    print("\nLoading test data...")
    X_test, y_test = load_images_from_path(os.path.join(BASE_PATH, 'chest_xray/test'), img_size)

    if len(X_train) == 0 or len(X_test) == 0 or len(X_val) == 0:
        raise ValueError("No images were loaded. Please check the paths and image files.")

    # Normalize the data
    X_train = X_train.astype('float32') / 255.0
    X_val = X_val.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # Combine all data for cross-validation
    X_all = np.concatenate((X_train, X_val, X_test))
    y_all = np.concatenate((y_train, y_val, y_test))

    # Label encoding
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(y_all)

    return X_all, y_all, label_encoder

if __name__ == "__main__":
    verify_paths()
    X_all, y_all, label_encoder = load_xray_data()

Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train/NORMAL
Number of images: 1341
Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train/PNEUMONIA
Number of images: 3875
Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/test/NORMAL
Number of images: 234
Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/test/PNEUMONIA
Number of images: 390
Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/val/NORMAL
Number of images: 8
Found directory: /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/val/PNEUMONIA
Number of images: 8

Loading training data...
Loading NORMAL images from /root/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versi

### STANDARD CNN

In [ ]:
#Standard CNN
def build_standard_cnn(input_shape, num_classes=2):
    model = models.Sequential()
    model.add(Input(shape=input_shape))
    model.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    return model

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:])
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.7914 - loss: 0.5125 - val_accuracy: 0.9403 - val_loss: 0.1495
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9345 - loss: 0.1625 - val_accuracy: 0.9539 - val_loss: 0.1302
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9582 - loss: 0.1163 - val_accuracy: 0.9590 - val_loss: 0.1220
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9599 - loss: 0.1091 - val_accuracy: 0.9437 - val_loss: 0.1484
Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9660 - loss: 0.0925 - val_accuracy: 0.9505 - val_loss: 0.1354
Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9729 - loss: 0.0719 - val_accuracy: 0.9573 - val_loss: 0.1235
Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9780 - loss: 0.0631 - val_accuracy: 0.9590 - val_loss: 0.1324
Epoch 8/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9853 - loss: 0.0515 

### Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes)
        recall = recall_score(y_val_fold, y_pred_classes)
        f1 = f1_score(y_val_fold, y_pred_classes)

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 54ms/step - accuracy: 0.7293 - loss: 0.6106 - val_accuracy: 0.7048 - val_loss: 0.6116
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.7280 - loss: 0.5874 - val_accuracy: 0.7048 - val_loss: 0.6095
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7266 - loss: 0.5858 - val_accuracy: 0.7048 - val_loss: 0.6079
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.7349 - loss: 0.5666 - val_accuracy: 0.6980 - val_loss: 0.5771
Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.7138 - loss: 0.5436 - val_accuracy: 0.7048 - val_loss: 0.5514
Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.7339 - loss: 0.5064 - val_accuracy: 0.7048 - val_loss: 0.5403
Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.7317 - loss: 0.4956 - val_accuracy: 0.7048 - val_loss: 0.5221
Epoch 8/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7300 - loss: 0.4904

### Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes)
        recall = recall_score(y_val_fold, y_pred_classes)
        f1 = f1_score(y_val_fold, y_pred_classes)

        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - accuracy: 0.7263 - loss: 0.5962 - val_accuracy: 0.7474 - val_loss: 0.5733
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7373 - loss: 0.5741 - val_accuracy: 0.7474 - val_loss: 0.5233
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7248 - loss: 0.5405 - val_accuracy: 0.7474 - val_loss: 0.4800
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7286 - loss: 0.4960 - val_accuracy: 0.7474 - val_loss: 0.4560
Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7304 - loss: 0.4900 - val_accuracy: 0.7526 - val_loss: 0.4623
Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7218 - loss: 0.4796 - val_accuracy: 0.7474 - val_loss: 0.4420
Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7412 - loss: 0.4774 - val_accuracy: 0.7833 - val_loss: 0.4388
Epoch 8/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7611 - loss: 0.4641 

### Arbitrary Granules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes)
        recall = recall_score(y_val_fold, y_pred_classes)
        f1 = f1_score(y_val_fold, y_pred_classes)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.8275 - loss: 0.3919 - val_accuracy: 0.8276 - val_loss: 0.3418
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8612 - loss: 0.3164 - val_accuracy: 0.7526 - val_loss: 0.6974
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8706 - loss: 0.3010 - val_accuracy: 0.7628 - val_loss: 0.5854
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8724 - loss: 0.2980 - val_accuracy: 0.8635 - val_loss: 0.3021
Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8898 - loss: 0.2624 - val_accuracy: 0.7782 - val_loss: 0.5686
Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8930 - loss: 0.2685 - val_accuracy: 0.8686 - val_loss: 0.3062
Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8903 - loss: 0.2563 - val_accuracy: 0.8345 - val_loss: 0.3681
Epoch 8/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8914 - loss: 0.2553 

# **Alzheimer's**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from skimage import filters
import time
from google.colab import drive

BASE_PATH = '/content/drive/MyDrive/Alzheimer_s Dataset'

def verify_paths():
    categories = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
    for split in ['train', 'test']:
        for category in categories:
            path = os.path.join(BASE_PATH, split, category)
            if not os.path.exists(path):
                print(f"Warning: Path does not exist: {path}")
            else:
                print(f"Found directory: {path}")
                print(f"Number of images: {len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])}")

def load_alzheimer_data(img_size=(128, 128)):
    categories = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

    def load_images_from_path(split_path, img_size):
        images = []
        labels = []
        for category in categories:
            category_path = os.path.join(split_path, category)
            if not os.path.exists(category_path):
                print(f"Warning: Path does not exist: {category_path}")
                continue

            print(f"Loading {category} images from {category_path}")
            image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in image_files:
                img_path = os.path.join(category_path, img_name)
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Failed to load image: {img_path}")
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    img = cv2.resize(img, img_size)
                    images.append(img)
                    labels.append(category)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

            print(f"Loaded {len(images)} images for {category}")

        return np.array(images), np.array(labels)

    print("\nLoading training data...")
    X_train, y_train = load_images_from_path(os.path.join(BASE_PATH, 'train'), img_size)

    print("\nLoading test data...")
    X_test, y_test = load_images_from_path(os.path.join(BASE_PATH, 'test'), img_size)

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError("No images were loaded. Please check the paths and image files.")

    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_test = label_encoder.transform(y_test)

    X_all = np.concatenate((X_train, X_test))
    y_all = np.concatenate((y_train, y_test))

    print("\nDataset Summary:")
    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    print(f"Image shape: {X_train[0].shape}")
    print(f"Classes: {label_encoder.classes_}")

    return X_all, y_all, label_encoder

if __name__ == "__main__":
    verify_paths()
    X_all, y_all, label_encoder = load_alzheimer_data()


Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/train/MildDemented
Number of images: 717
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/train/ModerateDemented
Number of images: 52
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/train/NonDemented
Number of images: 2560
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/train/VeryMildDemented
Number of images: 1792
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/test/MildDemented
Number of images: 179
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/test/ModerateDemented
Number of images: 12
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/test/NonDemented
Number of images: 640
Found directory: /content/drive/MyDrive/Alzheimer_s Dataset/test/VeryMildDemented
Number of images: 448

Loading training data...
Loading MildDemented images from /content/drive/MyDrive/Alzheimer_s Dataset/train/MildDemented
Loaded 717 images for MildDemented
Loading ModerateDemented i

### Standard CNN

In [ ]:
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # First convolutional layer with standard Conv2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Second convolutional layer
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Flatten the feature map
    x = layers.Flatten()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer with softmax activation
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:],num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.4380 - loss: 8.7359 - val_accuracy: 0.3906 - val_loss: 1.3922
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.4973 - loss: 1.2791 - val_accuracy: 0.5250 - val_loss: 1.2318
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5097 - loss: 1.2061 - val_accuracy: 0.5234 - val_loss: 1.1827
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.4916 - loss: 1.1680 - val_accuracy: 0.5234 - val_loss: 1.1496
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.4899 - loss: 1.1371 - val_accuracy: 0.5234 - val_loss: 1.1268
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4913 - loss: 1.1089 - val_accuracy: 0.5234 - val_loss: 1.1099
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.4900 - loss: 1.1015 - val_accuracy: 0.5234 - val_loss: 1.0979
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.5053 - loss: 1.0825 -

### Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_train, y_train, X_test, y_test, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_train, y_train, X_test, y_test, label_encoder)


Fold 1:
Epoch 1/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.5126 - loss: 1.1297 - val_accuracy: 0.3684 - val_loss: 1.0631
Epoch 2/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.5501 - loss: 0.9275 - val_accuracy: 0.4932 - val_loss: 1.1268
Epoch 3/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.5813 - loss: 0.8919 - val_accuracy: 0.4932 - val_loss: 1.1879
Epoch 4/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.5694 - loss: 0.8711 - val_accuracy: 0.4932 - val_loss: 1.9547
Epoch 5/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.5829 - loss: 0.8775 - val_accuracy: 0.5010 - val_loss: 1.0736
Epoch 6/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.5792 - loss: 0.8723 - val_accuracy: 0.4990 - val_loss: 1.4017
Epoch 7/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.5887 - loss: 0.8629 - val_accuracy: 0.5088 - val_loss: 0.9717
Epoch 8/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.5947 - loss: 0.8482 

### Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.5085 - loss: 1.1126 - val_accuracy: 0.3594 - val_loss: 1.2595
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.5655 - loss: 0.8986 - val_accuracy: 0.4812 - val_loss: 1.2669
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5671 - loss: 0.8898 - val_accuracy: 0.4859 - val_loss: 1.0955
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5643 - loss: 0.8775 - val_accuracy: 0.4797 - val_loss: 1.2508
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5841 - loss: 0.8587 - val_accuracy: 0.5344 - val_loss: 1.1613
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.6042 - loss: 0.8321 - val_accuracy: 0.3562 - val_loss: 1.3040
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5948 - loss: 0.8553 - val_accuracy: 0.4688 - val_loss: 1.0824
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.5999 - loss: 0.8473 -

### Arbitrary Granules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.4605 - loss: 1.1856 - val_accuracy: 0.4453 - val_loss: 1.0000
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5048 - loss: 1.0181 - val_accuracy: 0.5469 - val_loss: 0.9765
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.5074 - loss: 1.0027 - val_accuracy: 0.5500 - val_loss: 0.9793
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5189 - loss: 0.9870 - val_accuracy: 0.3375 - val_loss: 1.1986
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.5260 - loss: 0.9817 - val_accuracy: 0.5312 - val_loss: 0.9882
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5275 - loss: 0.9694 - val_accuracy: 0.5156 - val_loss: 1.0300
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5322 - loss: 0.9596 - val_accuracy: 0.5188 - val_loss: 1.3959
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.5400 - loss: 0.9641 

# **MNIST**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hojjatk/mnist-dataset")

print("Path to dataset files:", path)

100%|██████████| 22.0M/22.0M [00:00<00:00, 94.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1


In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from skimage import filters
import time
import struct
import gzip
import cv2


# Base path for dataset
BASE_PATH = '/root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1'

# File paths
TRAIN_IMAGES_PATH = os.path.join(BASE_PATH, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
TRAIN_LABELS_PATH = os.path.join(BASE_PATH, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
TEST_IMAGES_PATH = os.path.join(BASE_PATH, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
TEST_LABELS_PATH = os.path.join(BASE_PATH, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

# Function to verify dataset files exist
def verify_files():
    for path in [TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH, TEST_IMAGES_PATH, TEST_LABELS_PATH]:
        if not os.path.exists(path):
            print(f"Warning: File does not exist: {path}")
        else:
            print(f"Found file: {path}")

# Function to load MNIST data
def load_mnist_images(file_path, img_size=(28, 28)):
    with open(file_path, 'rb') as f:
        _, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num_images, rows, cols)

    # Resize images if needed
    images_resized = np.array([cv2.resize(img, img_size) for img in images])
    return images_resized

# Function to load MNIST labels
def load_mnist_labels(file_path):
    with open(file_path, 'rb') as f:
        _, num_labels = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

if __name__ == "__main__":
    verify_files()

    print("\nLoading training data...")
    X_train = load_mnist_images(TRAIN_IMAGES_PATH)
    y_train = load_mnist_labels(TRAIN_LABELS_PATH)

    print("\nLoading test data...")
    X_test = load_mnist_images(TEST_IMAGES_PATH)
    y_test = load_mnist_labels(TEST_LABELS_PATH)

    # Normalize images
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train[..., np.newaxis]  # Add channel dimension
    X_test = X_test[..., np.newaxis]

    # Combine all data for cross-validation
    X_all = np.concatenate((X_train, X_test))
    y_all = np.concatenate((y_train, y_test))

    # Label encoding
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(y_all)

    print("\nData loading complete.")

Found file: /root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1/train-images-idx3-ubyte/train-images-idx3-ubyte
Found file: /root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1/train-labels-idx1-ubyte/train-labels-idx1-ubyte
Found file: /root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte
Found file: /root/.cache/kagglehub/datasets/hojjatk/mnist-dataset/versions/1/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte

Loading training data...

Loading test data...

Data loading complete.


### STANDARD CNN

In [ ]:
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # First convolutional layer with standard Conv2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Second convolutional layer
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Flatten the feature map
    x = layers.Flatten()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer with softmax activation
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:],num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.8698 - loss: 0.4371 - val_accuracy: 0.9796 - val_loss: 0.0739
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.9686 - loss: 0.1090 - val_accuracy: 0.9880 - val_loss: 0.0387
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9784 - loss: 0.0771 - val_accuracy: 0.9867 - val_loss: 0.0444
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9804 - loss: 0.0715 - val_accuracy: 0.9896 - val_loss: 0.0406
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9836 - loss: 0.0549 - val_accuracy: 0.9871 - val_loss: 0.0466
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9871 - loss: 0.0438 - val_accuracy: 0.9893 - val_loss: 0.0490
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9866 - loss: 0.0450 - val_accuracy: 0.9913 - val_loss: 0.0342
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9885 - l

### Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.2303 - loss: 2.0218 - val_accuracy: 0.3549 - val_loss: 1.6486
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.3737 - loss: 1.6236 - val_accuracy: 0.4129 - val_loss: 1.5218
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.4139 - loss: 1.5154 - val_accuracy: 0.4429 - val_loss: 1.4517
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.4394 - loss: 1.4608 - val_accuracy: 0.4710 - val_loss: 1.3902
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.4648 - loss: 1.4064 - val_accuracy: 0.4911 - val_loss: 1.3536
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.4888 - loss: 1.3512 - val_accuracy: 0.5096 - val_loss: 1.3011
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.5029 - loss: 1.3169 - val_accuracy: 0.5331 - val_loss: 1.2603
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.5116 - l

### Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.5795 - loss: 1.2243 - val_accuracy: 0.6691 - val_loss: 0.9438
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8010 - loss: 0.6118 - val_accuracy: 0.6409 - val_loss: 1.1892
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8346 - loss: 0.5124 - val_accuracy: 0.6783 - val_loss: 0.9818
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8525 - loss: 0.4583 - val_accuracy: 0.7910 - val_loss: 0.6121
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8634 - loss: 0.4313 - val_accuracy: 0.5883 - val_loss: 1.4940
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8710 - loss: 0.4043 - val_accuracy: 0.8103 - val_loss: 0.6216
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8738 - loss: 0.3948 - val_accuracy: 0.6561 - val_loss: 1.6150
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8775 - loss

### Arbitrary Granules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.6064 - loss: 1.1575 - val_accuracy: 0.8246 - val_loss: 0.5403
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9039 - loss: 0.3157 - val_accuracy: 0.8900 - val_loss: 0.3622
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9313 - loss: 0.2295 - val_accuracy: 0.9260 - val_loss: 0.2215
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9427 - loss: 0.1902 - val_accuracy: 0.8460 - val_loss: 0.6478
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9495 - loss: 0.1659 - val_accuracy: 0.9327 - val_loss: 0.2086
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9546 - loss: 0.1488 - val_accuracy: 0.7930 - val_loss: 0.7319
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9569 - loss: 0.1387 - val_accuracy: 0.9516 - val_loss: 0.1529
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9591 - lo

# **Brain Tumour MRI**

In [ ]:
import os
import numpy as np
import cv2
from sklearn.preprocessing import LabelEncoder

# Define dataset path
BASE_PATH = '/root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1'

# Function to verify dataset structure
def verify_paths():
    categories = ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
    for category in categories:
        path = os.path.join(BASE_PATH, category)
        if not os.path.exists(path):
            print(f"Warning: Path does not exist: {path}")
        else:
            print(f"Found directory: {path}")
            print(f"Number of images: {len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])}")

# Function to load Medical MNIST dataset
def load_medical_mnist_data(img_size=(128, 128)):
    categories = ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
    images = []
    labels = []

    # Load images from dataset
    for category in categories:
        category_path = os.path.join(BASE_PATH, category)
        if not os.path.exists(category_path):
            print(f"Warning: Path does not exist: {category_path}")
            continue

        print(f"Loading {category} images from {category_path}")
        image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for img_name in image_files:
            img_path = os.path.join(category_path, img_name)
            try:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Load in grayscale
                if img is None:
                    print(f"Failed to load image: {img_path}")
                    continue
                img = cv2.resize(img, img_size)
                images.append(img)
                labels.append(category)
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")

        print(f"Loaded {len(images)} images for {category}")

    # Convert to NumPy arrays
    X_all = np.array(images, dtype='float32') / 255.0  # Normalize pixel values
    X_all = X_all[..., np.newaxis]  # Add channel dimension

    # Encode labels
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(labels)

    print("\nDataset Summary:")
    print(f"Total samples: {len(X_all)}")
    print(f"Image shape: {X_all[0].shape}")
    print(f"Classes: {label_encoder.classes_}")

    return X_all, y_all, label_encoder

if __name__ == "__main__":
    verify_paths()
    X_all, y_all, label_encoder = load_medical_mnist_data()

Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/AbdomenCT
Number of images: 10000
Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/BreastMRI
Number of images: 8954
Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/CXR
Number of images: 10000
Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/ChestCT
Number of images: 10000
Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/Hand
Number of images: 10000
Found directory: /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/HeadCT
Number of images: 10000
Loading AbdomenCT images from /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/AbdomenCT
Loaded 10000 images for AbdomenCT
Loading BreastMRI images from /root/.cache/kagglehub/datasets/andrewmvd/medical-mnist/versions/1/BreastMRI
Loaded 18954 images for BreastMRI
Loading CXR images from /ro

### Standard CNN

In [ ]:
#Standard CNN
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Flatten()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_all.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 65ms/step - accuracy: 0.4099 - loss: 9.1149 - val_accuracy: 0.2783 - val_loss: 15.2285
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.4047 - loss: 1.2491 - val_accuracy: 0.2844 - val_loss: 32.4093
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4456 - loss: 1.1397 - val_accuracy: 0.2875 - val_loss: 27.3910
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4425 - loss: 1.1087 - val_accuracy: 0.3792 - val_loss: 7.3569
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.3998 - loss: 1.2108 - val_accuracy: 0.4679 - val_loss: 2.1021
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.4188 - loss: 1.1813 - val_accuracy: 0.5229 - val_loss: 1.3347
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.4538 - loss: 1.1387 - val_accuracy: 0.5872 - val_loss: 1.2798
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5127 - loss: 1.1060 - val_accurac

Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.4397 - loss: 1.2394 - val_accuracy: 0.3089 - val_loss: 1.3563
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5735 - loss: 1.0536 - val_accuracy: 0.2569 - val_loss: 1.3588
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5667 - loss: 1.0068 - val_accuracy: 0.2630 - val_loss: 1.4312
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.6364 - loss: 0.9019 - val_accuracy: 0.4067 - val_loss: 1.4338
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.6540 - loss: 0.8824 - val_accuracy: 0.4190 - val_loss: 1.5812
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.6546 - loss: 0.8477 - val_accuracy: 0.4679 - val_loss: 1.1473
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.6386 - loss: 0.8559 - val_accuracy: 0.4924 - val_loss: 1.2792
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.6750 - loss: 0.8186 - val_accuracy: 

Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 83ms/step - accuracy: 0.4706 - loss: 1.2036 - val_accuracy: 0.2936 - val_loss: 1.3676
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5780 - loss: 1.0129 - val_accuracy: 0.2936 - val_loss: 1.5210
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.6241 - loss: 0.9191 - val_accuracy: 0.2936 - val_loss: 2.0559
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6239 - loss: 0.8682 - val_accuracy: 0.2936 - val_loss: 2.5075
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6564 - loss: 0.8472 - val_accuracy: 0.2936 - val_loss: 2.7629
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6746 - loss: 0.7738 - val_accuracy: 0.3486 - val_loss: 2.0026
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6574 - loss: 0.7923 - val_accuracy: 0.6239 - val_loss: 0.8936
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6950 - loss: 0.7857 - val_accuracy: 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.4805 - loss: 1.1938 - val_accuracy: 0.2599 - val_loss: 1.3977
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.5835 - loss: 1.0027 - val_accuracy: 0.2630 - val_loss: 1.5811
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6129 - loss: 0.9228 - val_accuracy: 0.2997 - val_loss: 1.6811
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6312 - loss: 0.8546 - val_accuracy: 0.2599 - val_loss: 2.4960
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6434 - loss: 0.8326 - val_accuracy: 0.2599 - val_loss: 3.0955
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6518 - loss: 0.8215 - val_accuracy: 0.3089 - val_loss: 2.1335
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.6710 - loss: 0.7866 - val_accuracy: 0.6147 - val_loss: 0.9851
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.6894 - loss: 0.7589 - val_accuracy: 0.6820 - v

### Arbitary Granules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Define CNN model
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Reshape inputs correctly without unpacking in graph mode
        input_shape = tf.shape(inputs)
        batch_size, h, w, c = input_shape[0], input_shape[1], input_shape[2], input_shape[3]

        # Reshape for convolution operation
        x = tf.reshape(inputs, [-1, h, w, c])

        # Apply convolution
        x = self.conv(x)

        # Reshape back to the original granulated format
        x = tf.reshape(x, [batch_size, h, w, self.filters])
        return x

def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.3881 - loss: 1.3033 - val_accuracy: 0.3792 - val_loss: 1.3133
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4558 - loss: 1.2052 - val_accuracy: 0.4190 - val_loss: 1.2239
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.4887 - loss: 1.1720 - val_accuracy: 0.4587 - val_loss: 1.2268
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4848 - loss: 1.1598 - val_accuracy: 0.4618 - val_loss: 1.1740
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5010 - loss: 1.1090 - val_accuracy: 0.4220 - val_loss: 1.2661
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5090 - loss: 1.0950 - val_accuracy: 0.4404 - val_loss: 1.2116
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5262 - loss: 1.0681 - val_accuracy: 0.4557 - val_loss: 1.2301
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5352 - loss: 1.0604 - val_accuracy: 0

# **Fashion MNIST**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

100%|██████████| 68.8M/68.8M [00:04<00:00, 17.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4


In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from skimage import filters
import time
import struct
import gzip
import cv2


# Base path for dataset
BASE_PATH = '/root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4'

# File paths
TRAIN_IMAGES_PATH = os.path.join(BASE_PATH, 'train-images-idx3-ubyte')
TRAIN_LABELS_PATH = os.path.join(BASE_PATH, 'train-labels-idx1-ubyte')
TEST_IMAGES_PATH = os.path.join(BASE_PATH, 't10k-images-idx3-ubyte')
TEST_LABELS_PATH = os.path.join(BASE_PATH, 't10k-labels-idx1-ubyte')

# Function to verify dataset files exist
def verify_files():
    for path in [TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH, TEST_IMAGES_PATH, TEST_LABELS_PATH]:
        if not os.path.exists(path):
            print(f"Warning: File does not exist: {path}")
        else:
            print(f"Found file: {path}")

# Function to load MNIST data
def load_mnist_images(file_path, img_size=(28, 28)):
    with open(file_path, 'rb') as f:
        _, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num_images, rows, cols)

    # Resize images if needed
    images_resized = np.array([cv2.resize(img, img_size) for img in images])
    return images_resized

# Function to load MNIST labels
def load_mnist_labels(file_path):
    with open(file_path, 'rb') as f:
        _, num_labels = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

if __name__ == "__main__":
    verify_files()

    print("\nLoading training data...")
    X_train = load_mnist_images(TRAIN_IMAGES_PATH)
    y_train = load_mnist_labels(TRAIN_LABELS_PATH)

    print("\nLoading test data...")
    X_test = load_mnist_images(TEST_IMAGES_PATH)
    y_test = load_mnist_labels(TEST_LABELS_PATH)

    # Normalize images
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train[..., np.newaxis]  # Add channel dimension
    X_test = X_test[..., np.newaxis]

    # Combine all data for cross-validation
    X_all = np.concatenate((X_train, X_test))
    y_all = np.concatenate((y_train, y_test))

    # Label encoding
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(y_all)
    print("\nDataset Summary:")
    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    print(f"Image shape: {X_train[0].shape}")
    print(f"Classes: {label_encoder.classes_}")
    print("\nData loading complete.")

Found file: /root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4/train-images-idx3-ubyte
Found file: /root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4/train-labels-idx1-ubyte
Found file: /root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4/t10k-images-idx3-ubyte
Found file: /root/.cache/kagglehub/datasets/zalando-research/fashionmnist/versions/4/t10k-labels-idx1-ubyte

Loading training data...

Loading test data...

Dataset Summary:
Training samples: 60000
Test samples: 10000
Image shape: (28, 28, 1)
Classes: [0 1 2 3 4 5 6 7 8 9]

Data loading complete.


### STANDARD CNN

In [ ]:
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # First convolutional layer with standard Conv2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Second convolutional layer
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Flatten the feature map
    x = layers.Flatten()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer with softmax activation
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:],num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.7599 - loss: 0.7082 - val_accuracy: 0.8933 - val_loss: 0.3112
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8640 - loss: 0.3826 - val_accuracy: 0.8883 - val_loss: 0.3074
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8850 - loss: 0.3165 - val_accuracy: 0.9141 - val_loss: 0.2465
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8965 - loss: 0.2900 - val_accuracy: 0.9047 - val_loss: 0.2575
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9071 - loss: 0.2586 - val_accuracy: 0.9146 - val_loss: 0.2468
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9129 - loss: 0.2360 - val_accuracy: 0.8960 - val_loss: 0.2899
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9171 - loss: 0.2267 - val_accuracy: 0.9004 - val_loss: 0.2760
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9213 - lo

### Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.2670 - loss: 1.8720 - val_accuracy: 0.5101 - val_loss: 1.2465
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.5306 - loss: 1.2185 - val_accuracy: 0.5560 - val_loss: 1.1449
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5748 - loss: 1.1225 - val_accuracy: 0.5989 - val_loss: 1.0782
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.6025 - loss: 1.0653 - val_accuracy: 0.6186 - val_loss: 1.0142
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.6178 - loss: 1.0214 - val_accuracy: 0.6264 - val_loss: 0.9933
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.6298 - loss: 0.9908 - val_accuracy: 0.6370 - val_loss: 0.9540
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.6414 - loss: 0.9612 - val_accuracy: 0.6419 - val_loss: 0.9441
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.6559 - los

### Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.5981 - loss: 1.1162 - val_accuracy: 0.7653 - val_loss: 0.6710
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7936 - loss: 0.5866 - val_accuracy: 0.7046 - val_loss: 0.8241
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8192 - loss: 0.5193 - val_accuracy: 0.7980 - val_loss: 0.5955
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8332 - loss: 0.4772 - val_accuracy: 0.8467 - val_loss: 0.4457
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.8453 - loss: 0.4444 - val_accuracy: 0.8186 - val_loss: 0.5088
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8483 - loss: 0.4284 - val_accuracy: 0.7910 - val_loss: 0.5578
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8531 - loss: 0.4118 - val_accuracy: 0.8620 - val_loss: 0.3982
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8554 - l

### Arbitrary Granules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.4838 - loss: 1.3797 - val_accuracy: 0.6189 - val_loss: 1.0457
Epoch 2/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.6696 - loss: 0.8918 - val_accuracy: 0.7187 - val_loss: 0.7684
Epoch 3/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.7157 - loss: 0.7724 - val_accuracy: 0.7317 - val_loss: 0.7348
Epoch 4/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7370 - loss: 0.7286 - val_accuracy: 0.7091 - val_loss: 0.7720
Epoch 5/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7527 - loss: 0.6774 - val_accuracy: 0.7400 - val_loss: 0.7061
Epoch 6/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7669 - loss: 0.6439 - val_accuracy: 0.6366 - val_loss: 1.0179
Epoch 7/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.7722 - loss: 0.6247 - val_accuracy: 0.7721 - val_loss: 0.6454
Epoch 8/10
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.7791 - l

# **Gastrointestinal diseases**

In [15]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("heartzhacker/medical-imaging")

print("Path to dataset files:", path)

100%|██████████| 673M/673M [00:11<00:00, 59.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1


In [16]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
import time
from skimage import filters

# Base path for the new dataset
BASE_PATH = '/root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset'

# Function to verify dataset structure
def verify_paths():
    categories = ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']
    for category in categories:
        path = os.path.join(BASE_PATH, category)
        if not os.path.exists(path):
            print(f"Warning: Path does not exist: {path}")
        else:
            print(f"Found directory: {path}")
            print(f"Number of images: {len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])}")

# Function to load images
def load_medical_data(img_size=(128, 128)):
    categories = ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']

    def load_images_from_path(base_path, img_size):
        images = []
        labels = []
        for category in categories:
            category_path = os.path.join(base_path, category)
            if not os.path.exists(category_path):
                print(f"Warning: Path does not exist: {category_path}")
                continue

            print(f"Loading {category} images from {category_path}")
            image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in image_files:
                img_path = os.path.join(category_path, img_name)
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Failed to load image: {img_path}")
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    img = cv2.resize(img, img_size)
                    images.append(img)
                    labels.append(category)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

            print(f"Loaded {len(images)} images for {category}")

        return np.array(images), np.array(labels)

    print("\nLoading dataset...")
    X_all, y_all = load_images_from_path(BASE_PATH, img_size)

    if len(X_all) == 0:
        raise ValueError("No images were loaded. Please check the dataset path.")

    # Normalize the data
    X_all = X_all.astype('float32') / 255.0
    X_all = X_all[..., np.newaxis]

    # Encode labels
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(y_all)

    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}, Test samples: {len(X_test)}")

    return X_all, y_all, label_encoder

if __name__ == "__main__":
    verify_paths()
    X_all, y_all, label_encoder = load_medical_data()


Found directory: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Diverticulosis
Number of images: 1000
Found directory: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Neoplasm
Number of images: 1000
Found directory: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Peritonitis
Number of images: 1000
Found directory: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Ureters
Number of images: 1000

Loading dataset...
Loading Diverticulosis images from /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Diverticulosis
Loaded 1000 images for Diverticulosis
Loading Neoplasm images from /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1/Medical-imaging-dataset/Neoplasm
Loaded 2000 images for Neoplasm
Loading Peritonitis images from /root/.cac

## Standard CNN

In [ ]:
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # First convolutional layer with standard Conv2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Second convolutional layer
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Flatten the feature map
    x = layers.Flatten()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer with softmax activation
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:],num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - accuracy: 0.3313 - loss: 7.1460 - val_accuracy: 0.3400 - val_loss: 6.6455
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3617 - loss: 1.3597 - val_accuracy: 0.3500 - val_loss: 7.1478
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.3484 - loss: 1.3596 - val_accuracy: 0.3725 - val_loss: 5.0812
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.3582 - loss: 1.3002 - val_accuracy: 0.4025 - val_loss: 1.5375
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.3482 - loss: 1.3068 - val_accuracy: 0.4600 - val_loss: 1.2390
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3394 - loss: 1.3010 - val_accuracy: 0.4650 - val_loss: 1.1801
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.3669 - loss: 1.2809 - val_accuracy: 0.4625 - val_loss: 1.1729
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3673 - loss: 1.2620 

### Fixed Granules CNN (3x3)

In [ ]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 15s 73ms/step - accuracy: 0.2537 - loss: 1.3869 - val_accuracy: 0.2200 - val_loss: 1.3867
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.2786 - loss: 1.3813 - val_accuracy: 0.3325 - val_loss: 1.3544
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.3548 - loss: 1.3411 - val_accuracy: 0.3350 - val_loss: 1.3074
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.3478 - loss: 1.3072 - val_accuracy: 0.3625 - val_loss: 1.2779
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.3612 - loss: 1.3124 - val_accuracy: 0.3425 - val_loss: 1.3014
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.3737 - loss: 1.2990 - val_accuracy: 0.3800 - val_loss: 1.2817
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.3812 - loss: 1.2991 - val_accuracy: 0.3875 - val_loss: 1.2688
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.3754 - loss: 1.2845

### Fixed Granules CNN (5x5)

In [ ]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average="macro")
        recall = recall_score(y_val_fold, y_pred_classes, average="macro")
        f1 = f1_score(y_val_fold, y_pred_classes, average="macro")
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - accuracy: 0.3782 - loss: 1.3059 - val_accuracy: 0.2625 - val_loss: 1.3807
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5339 - loss: 1.0876 - val_accuracy: 0.3175 - val_loss: 1.4881
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5794 - loss: 1.0014 - val_accuracy: 0.2925 - val_loss: 1.6175
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6282 - loss: 0.9412 - val_accuracy: 0.2625 - val_loss: 2.0728
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6323 - loss: 0.9058 - val_accuracy: 0.2625 - val_loss: 2.8767
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6575 - loss: 0.8664 - val_accuracy: 0.3325 - val_loss: 1.9546
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6435 - loss: 0.8688 - val_accuracy: 0.4050 - val_loss: 1.6285
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6553 - loss: 0.8392 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Fixed Granules CNN (5x5) Fold Results: Accuracy=0.5925, Precision=0.4451, Recall=0.6138, F1=0.5129

Fold 7:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.4142 - loss: 1.2970 - val_accuracy: 0.2700 - val_loss: 1.4772
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5621 - loss: 1.0410 - val_accuracy: 0.2700 - val_loss: 2.7781
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5829 - loss: 0.9857 - val_accuracy: 0.2700 - val_loss: 3.6463
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6034 - loss: 0.9551 - val_accuracy: 0.2700 - val_loss: 3.5178
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6258 - loss: 0.8918 - val_accuracy: 0.2750 - val_loss: 3.3495
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6501 - loss: 0.8344 - val_accuracy: 0.3525 - val_loss: 1.9251
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6652 - loss: 0.8280 - val_accuracy: 0.5650 - val_

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Fixed Granules CNN (5x5) Fold Results: Accuracy=0.4750, Precision=0.3689, Recall=0.4726, F1=0.3938

Fold 10:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.3806 - loss: 1.3090 - val_accuracy: 0.2975 - val_loss: 1.3838
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5377 - loss: 1.0817 - val_accuracy: 0.2975 - val_loss: 2.3453
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5742 - loss: 1.0029 - val_accuracy: 0.2975 - val_loss: 3.1945
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6064 - loss: 0.9364 - val_accuracy: 0.2975 - val_loss: 2.7238
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6364 - loss: 0.8753 - val_accuracy: 0.4050 - val_loss: 1.7411
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6481 - loss: 0.8664 - val_accuracy: 0.5000 - val_loss: 1.3034
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6469 - loss: 0.8543 - val_accuracy: 0.3700 - val_

### Arbitrary Grabules CNN

In [ ]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X_all, y_all, label_encoder)


Fold 1:
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - accuracy: 0.3129 - loss: 1.3589 - val_accuracy: 0.3525 - val_loss: 1.3757
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.3600 - loss: 1.3210 - val_accuracy: 0.3575 - val_loss: 1.3299
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3883 - loss: 1.2963 - val_accuracy: 0.3950 - val_loss: 1.3260
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4295 - loss: 1.2525 - val_accuracy: 0.4075 - val_loss: 1.2613
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4430 - loss: 1.2425 - val_accuracy: 0.4450 - val_loss: 1.2313
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4591 - loss: 1.2095 - val_accuracy: 0.4475 - val_loss: 1.2471
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4774 - loss: 1.1707 - val_accuracy: 0.4200 - val_loss: 1.2185
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4961 - loss: 1.1435 

# **Tuberculosis**

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")

print("Path to dataset files:", path)

100%|██████████| 663M/663M [00:03<00:00, 196MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3


In [6]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold
from skimage import filters
import time
from sklearn.model_selection import train_test_split

# Base path for the TB dataset
BASE_PATH = '/root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database'

# Categories for classification
CATEGORIES = ['Normal', 'Tuberculosis']

# Function to verify dataset paths
def verify_paths():
    for category in CATEGORIES:
        path = os.path.join(BASE_PATH, category)
        if not os.path.exists(path):
            print(f"Warning: Path does not exist: {path}")
        else:
            print(f"Found directory: {path}")
            print(f"Number of images: {len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])}")

# Function to load images
def load_xray_data(img_size=(128, 128)):
    images, labels = [], []

    for category in CATEGORIES:
        category_path = os.path.join(BASE_PATH, category)
        if not os.path.exists(category_path):
            print(f"Warning: Path does not exist: {category_path}")
            continue

        print(f"Loading {category} images from {category_path}")
        image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for img_name in image_files:
            img_path = os.path.join(category_path, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Failed to load image: {img_path}")
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                img = cv2.resize(img, img_size)
                images.append(img)
                labels.append(category)
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")

    images = np.array(images).astype('float32') / 255.0
    images = images[..., np.newaxis]

    # Label encoding
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(labels)

    return images, labels, label_encoder

if __name__ == "__main__":
    verify_paths()
    X, y, label_encoder = load_xray_data()

    # Split dataset into train, validation, and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}, Test samples: {len(X_test)}")


Found directory: /root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Normal
Number of images: 3500
Found directory: /root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Tuberculosis
Number of images: 700
Loading Normal images from /root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Normal
Loading Tuberculosis images from /root/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Tuberculosis
Train samples: 2688, Validation samples: 672, Test samples: 840


## Standard CNN

In [9]:
def build_standard_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    # First convolutional layer with standard Conv2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Second convolutional layer
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Flatten the feature map
    x = layers.Flatten()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer with softmax activation
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

standard_cnn_results = []
total_time_standard = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    standard_cnn_results = []
    total_time_standard = 0

    # Initialize lists to store results for each model

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Standard CNN
        model = build_standard_cnn(X_train_fold.shape[1:],num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_standard += end_time - start_time

        y_pred = model.predict(X_val_fold)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes,average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))
        print(f"Standard CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    standard_cnn_mean_results = np.mean(standard_cnn_results, axis=0)
    standard_cnn_std_results = np.std(standard_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Standard CNN: Accuracy={standard_cnn_mean_results[0]:.4f} ± {standard_cnn_std_results[0]:.4f}, Precision={standard_cnn_mean_results[1]:.4f} ± {standard_cnn_std_results[1]:.4f}, Recall={standard_cnn_mean_results[2]:.4f} ± {standard_cnn_std_results[2]:.4f}, F1={standard_cnn_mean_results[3]:.4f} ± {standard_cnn_std_results[3]:.4f}")


    print(f"\nTotal Training Time:")
    print(f"Standard CNN: {total_time_standard:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X, y, label_encoder)



Fold 1:
Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.8804 - loss: 2.5102 - val_accuracy: 0.8429 - val_loss: 1.7668
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9583 - loss: 0.2064 - val_accuracy: 0.8429 - val_loss: 3.7397
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9657 - loss: 0.2067 - val_accuracy: 0.8786 - val_loss: 1.7707
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9643 - loss: 0.2906 - val_accuracy: 0.9452 - val_loss: 0.3384
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9677 - loss: 0.1158 - val_accuracy: 0.9214 - val_loss: 0.4119
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9750 - loss: 0.0943 - val_accuracy: 0.8452 - val_loss: 0.5034
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9780 - loss: 0.0762 - val_accuracy: 0.9619 - val_loss: 0.3213
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9853 - loss: 0.0520 

### Fixed Granules CNN (3x3)

In [12]:
#Fixed Granules CNN (3x3)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_3x3 = []
total_time_fixed_granules_3x3 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    fixed_granules_cnn_results_3x3 = []
    total_time_fixed_granules_3x3 = 0
    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(3,3))
        X_val_granulated = apply_granulation(X_val_fold,(3,3))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_3x3 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        standard_cnn_results.append((accuracy, precision, recall, f1))

        fixed_granules_cnn_results_3x3.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (3x3) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_3x3 = np.mean(fixed_granules_cnn_results_3x3, axis=0)
    fixed_granules_cnn_std_results_3x3 = np.std(fixed_granules_cnn_results_3x3, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (3x3): Accuracy={fixed_granules_cnn_mean_results_3x3[0]:.4f} ± {fixed_granules_cnn_std_results_3x3[0]:.4f}, Precision={fixed_granules_cnn_mean_results_3x3[1]:.4f} ± {fixed_granules_cnn_std_results_3x3[1]:.4f}, Recall={fixed_granules_cnn_mean_results_3x3[2]:.4f} ± {fixed_granules_cnn_std_results_3x3[2]:.4f}, F1={fixed_granules_cnn_mean_results_3x3[3]:.4f} ± {fixed_granules_cnn_std_results_3x3[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (3x3): {total_time_fixed_granules_3x3:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X, y, label_encoder)


Fold 1:
Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - accuracy: 0.8050 - loss: 0.5274 - val_accuracy: 0.8357 - val_loss: 0.4452
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.8273 - loss: 0.4589 - val_accuracy: 0.8357 - val_loss: 0.4435
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.8317 - loss: 0.4507 - val_accuracy: 0.8357 - val_loss: 0.4438
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.8257 - loss: 0.4606 - val_accuracy: 0.8357 - val_loss: 0.4424
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.8369 - loss: 0.4409 - val_accuracy: 0.8357 - val_loss: 0.4420
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8337 - loss: 0.4460 - val_accuracy: 0.8357 - val_loss: 0.4419
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.8302 - loss: 0.4482 - val_accuracy: 0.8357 - val_loss: 0.4436
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.8358 - loss: 0.4399

### Fixed Granules CNN (5x5)

In [13]:
#Fixed Granules CNN (5x5)
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with fixed granules
def build_optimized_cnn_fixed_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = GranularConv2D(32, (3, 3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)  # Use 3D MaxPooling to handle the granulation dimension
    x = GranularConv2D(64, (3, 3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)  # Use 3D global average pooling to aggregate over granules
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    return models.Model(inputs=inputs, outputs=outputs)

# Function to apply granulation with fixed granules
def apply_granulation(images, grain_size):
    batch_size, h, w, c = images.shape
    gh, gw = grain_size
    pad_h = (-(h % -gh)) % gh
    pad_w = (-(w % -gw)) % gw

    # Padding images if necessary
    if pad_h > 0 or pad_w > 0:
        images = tf.pad(images, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        h, w = h + pad_h, w + pad_w

    num_grains_h, num_grains_w = h // gh, w // gw
    grains = tf.image.extract_patches(
        images,
        sizes=[1, gh, gw, 1],
        strides=[1, gh, gw, 1],
        rates=[1, 1, 1, 1],
        padding="VALID"
    )
    return tf.reshape(grains, [batch_size, num_grains_h * num_grains_w, gh, gw, c])

fixed_granules_cnn_results_5x5 = []
total_time_fixed_granules_5x5 = 0

# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global fixed_granules_cnn_results_5x5
    global total_time_fixed_granules_5x5

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Fixed Granules CNN
        X_train_granulated = apply_granulation(X_train_fold,(5,5))
        X_val_granulated = apply_granulation(X_val_fold,(5,5))

        model = build_optimized_cnn_fixed_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_fixed_granules_5x5 += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        fixed_granules_cnn_results_5x5.append((accuracy, precision, recall, f1))
        print(f"\nFixed Granules CNN (5x5) Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    fixed_granules_cnn_mean_results_5x5 = np.mean(fixed_granules_cnn_results_5x5, axis=0)
    fixed_granules_cnn_std_results_5x5 = np.std(fixed_granules_cnn_results_5x5, axis=0)

    print("\nMean Results:")
    print(f"Fixed Granules CNN (5x5): Accuracy={fixed_granules_cnn_mean_results_5x5[0]:.4f} ± {fixed_granules_cnn_std_results_5x5[0]:.4f}, Precision={fixed_granules_cnn_mean_results_5x5[1]:.4f} ± {fixed_granules_cnn_std_results_5x5[1]:.4f}, Recall={fixed_granules_cnn_mean_results_5x5[2]:.4f} ± {fixed_granules_cnn_std_results_5x5[2]:.4f}, F1={fixed_granules_cnn_mean_results_5x5[3]:.4f} ± {fixed_granules_cnn_std_results_5x5[3]:.4f}")

    print(f"\nTotal Training Time:")
    print(f"Fixed Granules CNN (5x5): {total_time_fixed_granules_5x5:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X, y, label_encoder)


Fold 1:
Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.8054 - loss: 0.5413 - val_accuracy: 0.8167 - val_loss: 0.5097
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.8756 - loss: 0.3272 - val_accuracy: 0.1833 - val_loss: 1.3880
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9078 - loss: 0.2491 - val_accuracy: 0.1833 - val_loss: 2.3422
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9192 - loss: 0.2067 - val_accuracy: 0.1833 - val_loss: 3.4414
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9454 - loss: 0.1598 - val_accuracy: 0.7310 - val_loss: 0.5419
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9416 - loss: 0.1664 - val_accuracy: 0.2786 - val_loss: 1.5651
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9696 - loss: 0.1057 - val_accuracy: 0.2643 - val_loss: 1.5935
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9582 - loss: 0.1257 

### Arbitrary Grabules CNN

In [8]:
# Function to calculate intensity differences (proxy for color differences)
def calculate_intensity_differences(images):
    batch_size, h, w, c = images.shape
    differences = []

    for i in range(batch_size):
        image = images[i].squeeze()  # Remove channel dimension
        for y in range(h):
            for x in range(w):
                if x > 0 and y > 0:
                    current_intensity = image[y, x]
                    left_intensity = image[y, x-1]
                    top_intensity = image[y-1, x]
                    differences.append(np.abs(current_intensity - left_intensity))
                    differences.append(np.abs(current_intensity - top_intensity))

    return np.array(differences)

def otsu_threshold(differences):
    return filters.threshold_otsu(differences)

# Function to apply granulation with arbitrary granules
def region_growing_granulation(images, threshold):
    """
    Applies region-growing segmentation to form arbitrary-shaped granules.
    Args:
        images: Input batch of images (shape: batch_size, height, width, channels).
        threshold: Intensity nearness threshold for region growing.
    Returns:
        Granulated images with arbitrary-shaped and overlapping regions.
    """
    batch_size, h, w, c = images.shape
    granulated_images = np.zeros_like(images)  # Initialize output array

    for b in range(batch_size):
        image = images[b].squeeze()  # Remove channel dimension
        labels = np.zeros((h, w), dtype=int)  # Label map for regions
        current_label = 1  # Start labeling regions

        # Iterate through all pixels to grow regions
        for y in range(h):
            for x in range(w):
                if labels[y, x] == 0:  # If pixel is not labeled yet
                    # Start a new region from this pixel
                    seed_intensity = image[y, x]
                    region_pixels = [(y, x)]  # List of pixels in this region

                    while region_pixels:
                        py, px = region_pixels.pop()
                        labels[py, px] = current_label

                        # Check 4-neighbors for similarity
                        neighbors = [(py-1, px), (py+1, px), (py, px-1), (py, px+1)]
                        for ny, nx in neighbors:
                            if 0 <= ny < h and 0 <= nx < w and labels[ny, nx] == 0:
                                neighbor_intensity = image[ny, nx]
                                if np.abs(neighbor_intensity - seed_intensity) < threshold:
                                    labels[ny, nx] = current_label
                                    region_pixels.append((ny, nx))

                    current_label += 1

        # Map labeled regions back to the original image format
        granulated_images[b] = labels[..., np.newaxis]

    return granulated_images

# Custom GranularConv2D Layer
class GranularConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Applying a standard Conv2D on each patch
        self.conv = layers.Conv2D(self.filters, self.kernel_size, padding='same', activation='relu')

    def call(self, inputs):
        # Input is 5D with shape (batch_size, num_grains, h, w, c)
        batch_size, num_grains, h, w, c = tf.shape(inputs)[0], tf.shape(inputs)[1], tf.shape(inputs)[2], tf.shape(inputs)[3], tf.shape(inputs)[4]
        x = tf.reshape(inputs, [-1, h, w, c])  # Flatten the patch dimension
        x = self.conv(x)  # Apply convolution on patches
        x = tf.reshape(x, [batch_size, num_grains, h, w, self.filters])  # Reshape back to (batch_size, num_grains, h, w, filters)
        return x

# Function to build an optimized CNN model with arbitrary granules
def build_optimized_cnn_arbitrary_granules(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

arbitrary_granules_cnn_results = []
total_time_arbitrary_granules = 0
# Main function to perform cross-validation for each model
def perform_cross_validation(X_all, y_all, label_encoder):
    kf = KFold(n_splits=10, shuffle=True)
    global arbitrary_granules_cnn_results
    global total_time_arbitrary_granules

    for fold, (train_index, val_index) in enumerate(kf.split(X_all)):
        X_train_fold, X_val_fold = X_all[train_index], X_all[val_index]
        y_train_fold, y_val_fold = y_all[train_index], y_all[val_index]

        print(f"\nFold {fold+1}:")

        # Arbitrary Granules CNN
        differences = calculate_intensity_differences(X_train_fold)
        THR = otsu_threshold(differences)
        X_train_granulated = region_growing_granulation(X_train_fold, THR)
        X_val_granulated = region_growing_granulation(X_val_fold, THR)

        model = build_optimized_cnn_arbitrary_granules(X_train_granulated.shape[1:], num_classes=len(label_encoder.classes_))
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        model.fit(X_train_granulated, y_train_fold, validation_data=(X_val_granulated, y_val_fold), batch_size=32, epochs=10, verbose=1)
        end_time = time.time()
        total_time_arbitrary_granules += end_time - start_time

        y_pred = model.predict(X_val_granulated)
        y_pred_classes = np.argmax(y_pred, axis=1)
        accuracy = np.mean(y_pred_classes == y_val_fold)
        precision = precision_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        recall = recall_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)
        f1 = f1_score(y_val_fold, y_pred_classes, average='macro', zero_division=1)

        arbitrary_granules_cnn_results.append((accuracy, precision, recall, f1))
        print(f"\nArbitrary Granules CNN Fold Results: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

    arbitrary_granules_cnn_mean_results = np.mean(arbitrary_granules_cnn_results, axis=0)
    arbitrary_granules_cnn_std_results = np.std(arbitrary_granules_cnn_results, axis=0)

    print("\nMean Results:")
    print(f"Arbitrary Granules CNN: Accuracy={arbitrary_granules_cnn_mean_results[0]:.4f} ± {arbitrary_granules_cnn_std_results[0]:.4f}, Precision={arbitrary_granules_cnn_mean_results[1]:.4f} ± {arbitrary_granules_cnn_std_results[1]:.4f}, Recall={arbitrary_granules_cnn_mean_results[2]:.4f} ± {arbitrary_granules_cnn_std_results[2]:.4f}, F1={arbitrary_granules_cnn_mean_results[3]:.4f} ± {arbitrary_granules_cnn_std_results[3]:.4f}")
    print("\nTraining Time:")
    print(f"Arbitrary Granules CNN: {total_time_arbitrary_granules:.2f} seconds")

if __name__ == "__main__":
    perform_cross_validation(X, y, label_encoder)


Fold 1:
Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.8288 - loss: 0.4217 - val_accuracy: 0.8714 - val_loss: 0.3475
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8823 - loss: 0.2762 - val_accuracy: 0.8738 - val_loss: 0.3024
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8901 - loss: 0.2562 - val_accuracy: 0.8976 - val_loss: 0.2670
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9160 - loss: 0.2180 - val_accuracy: 0.8548 - val_loss: 0.3250
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9186 - loss: 0.2036 - val_accuracy: 0.9333 - val_loss: 0.1793
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9183 - loss: 0.1946 - val_accuracy: 0.9095 - val_loss: 0.1822
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9338 - loss: 0.1654 - val_accuracy: 0.9357 - val_loss: 0.1568
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9320 - loss: 0.1633 